# Fold 1 EGNN Analysis

This notebook tests whether cached ESM embeddings carry signal for inactive → active conformational change prediction, independently of FoldFlow.
It trains two small EGNN baselines on the existing Fold 1 pairs:
- `EGNN_delta_only`
- `EGNN_delta_plus_distance`

The models predict residue-wise coordinate deltas from inactive Cα coordinates plus ESM embeddings.

In [1]:
import json
import math
import os
import random
import sys
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(x, **kwargs):
        return x


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'kinase_data').is_dir() and (candidate / 'notebooks').is_dir():
            return candidate
    raise FileNotFoundError('Could not locate the repository root')


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

from kinase_data.translation import InactiveActiveTranslationDataset

REPORT_ROOT = PROJECT_ROOT / 'reports' / 'fold1_egnn'
FIGURE_ROOT = PROJECT_ROOT / 'figures' / 'fold1_egnn'
CHECKPOINT_ROOT = PROJECT_ROOT / 'checkpoints' / 'fold1_egnn_analysis'
for path in (REPORT_ROOT, FIGURE_ROOT, CHECKPOINT_ROOT):
    path.mkdir(parents=True, exist_ok=True)

SEED = 7
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
use_amp = device.type == 'cuda'

print('Project root:', PROJECT_ROOT)
print('Device:', device)
print('CUDA available:', torch.cuda.is_available())
print('AMP enabled:', use_amp)
print('Report root:', REPORT_ROOT)
print('Figure root:', FIGURE_ROOT)
print('Checkpoint root:', CHECKPOINT_ROOT)


def safe_json_write(path: Path, payload: dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2) + '\n', encoding='utf-8')


def process_memory_mb() -> float:
    try:
        import psutil
        return float(psutil.Process(os.getpid()).memory_info().rss / 1e6)
    except Exception:
        import resource
        rss = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
        if sys.platform == 'darwin':
            return float(rss / (1024 * 1024))
        return float(rss / 1024)


print('Memory before loading data (MB):', round(process_memory_mb(), 2))

Project root: /Users/josefinadehan/tp_vision_avanzada/TPF_Vision_Avanzada
Device: cpu
CUDA available: False
AMP enabled: False
Report root: /Users/josefinadehan/tp_vision_avanzada/TPF_Vision_Avanzada/reports/fold1_egnn
Figure root: /Users/josefinadehan/tp_vision_avanzada/TPF_Vision_Avanzada/figures/fold1_egnn
Checkpoint root: /Users/josefinadehan/tp_vision_avanzada/TPF_Vision_Avanzada/checkpoints/fold1_egnn_analysis
Memory before loading data (MB): 303.4


/Users/josefinadehan/miniforge3/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def structure_ca_coordinates(structure: dict[str, torch.Tensor] | dict[str, Any]) -> np.ndarray:
    coords = structure['rigids']
    if isinstance(coords, torch.Tensor):
        coords = coords.detach().cpu().numpy()
    coords = np.asarray(coords, dtype=np.float64)
    if coords.ndim == 3:
        coords = coords[0]
    return coords[..., 4:7]


def rmsd_from_coordinates(reference: np.ndarray, moving: np.ndarray, align: bool = True) -> float:
    reference = np.asarray(reference, dtype=np.float64)
    moving = np.asarray(moving, dtype=np.float64)
    if reference.shape != moving.shape or reference.ndim != 2 or reference.shape[1] != 3:
        raise ValueError('Coordinates must have shape [N, 3] and match in size')
    if reference.size == 0:
        return float('nan')
    if not np.isfinite(reference).all() or not np.isfinite(moving).all():
        return float('inf')
    if not align:
        diff = reference - moving
        return float(np.sqrt(np.mean(np.sum(diff * diff, axis=-1))))
    reference_centered = reference - reference.mean(axis=0, keepdims=True)
    moving_centered = moving - moving.mean(axis=0, keepdims=True)
    covariance = moving_centered.T @ reference_centered
    try:
        u, _, vt = np.linalg.svd(covariance)
    except np.linalg.LinAlgError:
        diff = reference_centered - moving_centered
        return float(np.sqrt(np.mean(np.sum(diff * diff, axis=-1))))
    rotation = u @ vt
    if np.linalg.det(rotation) < 0:
        vt[-1, :] *= -1
        rotation = u @ vt
    aligned = moving_centered @ rotation
    diff = reference_centered - aligned
    return float(np.sqrt(np.mean(np.sum(diff * diff, axis=-1))))


def one_hot_aatype(aatype: torch.Tensor, num_classes: int = 21) -> torch.Tensor:
    return F.one_hot(aatype.clamp(min=0, max=num_classes - 1).long(), num_classes=num_classes).float()


def pairwise_distances(coords: torch.Tensor, mask: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    diff = coords[:, :, None, :] - coords[:, None, :, :]
    dist = torch.sqrt(diff.pow(2).sum(dim=-1).clamp_min(1e-8))
    pair_mask = mask[:, :, None] * mask[:, None, :]
    eye = torch.eye(coords.shape[1], device=coords.device, dtype=pair_mask.dtype).unsqueeze(0)
    pair_mask = pair_mask * (1.0 - eye)
    return dist, pair_mask


def masked_mse(pred: torch.Tensor, target: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    mask = mask.float().unsqueeze(-1)
    return ((pred - target).pow(2) * mask).sum() / mask.sum().clamp_min(1.0) / 3.0


def masked_pairwise_mse(pred_coords: torch.Tensor, target_coords: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    pred_dist, pair_mask = pairwise_distances(pred_coords, mask)
    target_dist, _ = pairwise_distances(target_coords, mask)
    return (((pred_dist - target_dist).pow(2)) * pair_mask).sum() / pair_mask.sum().clamp_min(1.0)


def tensor_health(name: str, tensor: torch.Tensor) -> str:
    tensor = tensor.detach()
    if tensor.numel() == 0:
        return f'{name}: empty'
    finite = torch.isfinite(tensor)
    finite_ratio = float(finite.float().mean().item())
    if finite.any():
        values = tensor[finite]
        return (
            f"{name}: shape={tuple(tensor.shape)} finite={finite_ratio:.4f} "
            f"min={float(values.min()):.4f} max={float(values.max()):.4f} mean|x|={float(values.abs().mean()):.4f}"
        )
    return f'{name}: shape={tuple(tensor.shape)} finite={finite_ratio:.4f} all_nonfinite'


def save_csv(path: Path, frame: pd.DataFrame) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path, index=False)


def motion_correlation(true_motion: np.ndarray, pred_motion: np.ndarray) -> float:
    true_motion = np.asarray(true_motion, dtype=np.float64)
    pred_motion = np.asarray(pred_motion, dtype=np.float64)
    if true_motion.size < 2 or pred_motion.size < 2:
        return float('nan')
    if np.std(true_motion) < 1e-8 or np.std(pred_motion) < 1e-8:
        return float('nan')
    return float(np.corrcoef(true_motion, pred_motion)[0, 1])

In [3]:
train_ds = InactiveActiveTranslationDataset(fold_id=1, split='train', load_structures=True)
val_ds = InactiveActiveTranslationDataset(fold_id=1, split='validation', load_structures=True)
test_ds = InactiveActiveTranslationDataset(fold_id=1, split='test', load_structures=True)

print('Dataset sizes:')
print('  train:', len(train_ds))
print('  validation:', len(val_ds))
print('  test:', len(test_ds))


def dataset_summary(ds: InactiveActiveTranslationDataset, split_name: str) -> pd.DataFrame:
    rows = []
    for sample in ds:
        source = structure_ca_coordinates(sample['source_structure'])
        target = structure_ca_coordinates(sample['target_structure'])
        rows.append({
            'split': split_name,
            'kinase': sample['kinase'],
            'source_pdb_id': sample['source_pdb_id'],
            'target_pdb_id': sample['target_pdb_id'],
            'residue_count': int(sample['residue_count']),
            'sequence_identity': float(sample['sequence_identity']),
            'rmsd_source_target': rmsd_from_coordinates(target, source),
            'mean_true_delta_norm': float(np.linalg.norm(target - source, axis=-1).mean()),
            'median_true_delta_norm': float(np.median(np.linalg.norm(target - source, axis=-1))),
        })
    frame = pd.DataFrame(rows)
    return frame

train_summary = dataset_summary(train_ds, 'train')
val_summary = dataset_summary(val_ds, 'validation')
test_summary = dataset_summary(test_ds, 'test')

summary_frame = pd.concat([train_summary, val_summary, test_summary], ignore_index=True)
summary_stats = summary_frame.groupby('split').agg(
    n_samples=('kinase', 'size'),
    avg_residue_count=('residue_count', 'mean'),
    avg_sequence_identity=('sequence_identity', 'mean'),
    avg_rmsd_source_target=('rmsd_source_target', 'mean'),
    median_rmsd_source_target=('rmsd_source_target', 'median'),
).reset_index()

kinase_counts = summary_frame.groupby(['split', 'kinase']).size().reset_index(name='count')

print('\nSplit summary:')
display(summary_stats)
print('\nKinase counts:')
display(kinase_counts.pivot(index='kinase', columns='split', values='count').fillna(0).astype(int))

identity_test = test_summary.copy()
identity_test['rmsd_prediction_target'] = identity_test['rmsd_source_target']
identity_test['rmsd_source_prediction'] = 0.0
identity_test['rmsd_improvement'] = 0.0
identity_test['success'] = False
identity_test['mean_true_motion'] = identity_test['mean_true_delta_norm']
identity_test['mean_pred_motion'] = 0.0
identity_test['motion_correlation'] = np.nan
save_csv(REPORT_ROOT / 'fold1_egnn_identity_test.csv', identity_test)

foldflow_report_path = PROJECT_ROOT / 'reports' / 'fold1' / 'failure_modes' / 'fold1_failure_modes_report.json'
foldflow_reference = None
if foldflow_report_path.is_file():
    with foldflow_report_path.open('r', encoding='utf-8') as handle:
        foldflow_payload = json.load(handle)
    foldflow_reference = foldflow_payload.get('summary', {}).get('conditional_mean_rmsd_prediction_target')
    print('\nReference FoldFlow RMSD:', foldflow_reference)
else:
    print('\nFoldFlow failure report not found; skipping contextual comparison.')

Dataset sizes:
  train: 215
  validation: 90
  test: 101


/Users/josefinadehan/tp_vision_avanzada/TPF_Vision_Avanzada/third_party/foldflow_official/openfold/utils/tensor_utils.py:74: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/autograd/python_variable_indexing.cpp:312.)
  return data[ranges]



Split summary:


,split,n_samples,avg_residue_count,avg_sequence_identity,avg_rmsd_source_target,median_rmsd_source_target
0,test,101,277.772277,0.987711,3.316142,3.172917
1,train,215,284.553488,0.989324,4.034038,4.771114
2,validation,90,261.122222,0.996974,3.077655,2.584227



Kinase counts:


split,test,train,validation
kinase,,,
ABL1,0,35,0
AKT1,0,17,0
ALK,0,3,0
BRAF,0,0,90
CDK4,0,11,0
CDK6,0,14,0
EGFR,101,0,0
ERBB2,0,3,0
FGFR1,0,2,0



Reference FoldFlow RMSD: None


In [4]:
@dataclass(frozen=True)
class EGNNAnalysisConfig:
    fold_id: int = 1
    hidden_dim: int = 96
    message_dim: int = 96
    num_layers: int = 3
    batch_size: int = 2
    max_epochs: int = 15
    patience: int = 4
    learning_rate: float = 5e-5
    weight_decay: float = 1e-4
    grad_clip: float = 1.0
    num_workers: int = 0
    coord_update_scale: float = 0.01
    delta_scale: float = 0.5
    lambda_distance: float = 0.05
    use_amp: bool = device.type == 'cuda'


class DeltaEGNNLayer(nn.Module):
    def __init__(self, hidden_dim: int, message_dim: int, coord_update_scale: float):
        super().__init__()
        edge_in = hidden_dim * 2 + 1
        self.edge_mlp = nn.Sequential(
            nn.Linear(edge_in, message_dim),
            nn.SiLU(),
            nn.Linear(message_dim, message_dim),
            nn.SiLU(),
        )
        self.node_mlp = nn.Sequential(
            nn.Linear(hidden_dim + message_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
        )
        self.coord_mlp = nn.Sequential(
            nn.Linear(edge_in, message_dim),
            nn.SiLU(),
            nn.Linear(message_dim, 1),
        )
        self.node_norm = nn.LayerNorm(hidden_dim)
        self.coord_update_scale = coord_update_scale

    def forward(self, h: torch.Tensor, coords: torch.Tensor, mask: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        nres = coords.shape[1]
        pair_mask = mask[:, :, None] * mask[:, None, :]
        eye = torch.eye(nres, device=coords.device, dtype=pair_mask.dtype).unsqueeze(0)
        pair_mask = pair_mask * (1.0 - eye)

        rel = coords[:, :, None, :] - coords[:, None, :, :]
        dist2 = rel.pow(2).sum(dim=-1, keepdim=True)
        hi = h[:, :, None, :].expand(-1, -1, nres, -1)
        hj = h[:, None, :, :].expand(-1, nres, -1, -1)
        edge_input = torch.cat([hi, hj, dist2], dim=-1)
        degree = pair_mask.sum(dim=2, keepdim=True).clamp_min(1.0)

        messages = self.edge_mlp(edge_input) * pair_mask.unsqueeze(-1)
        agg = messages.sum(dim=2) / degree
        h = self.node_norm(h + self.node_mlp(torch.cat([h, agg], dim=-1)))

        coord_weights = torch.tanh(self.coord_mlp(edge_input)) * pair_mask.unsqueeze(-1)
        coord_update = (coord_weights * rel).sum(dim=2) / degree
        coords = coords + self.coord_update_scale * coord_update
        coords = coords * mask.unsqueeze(-1)
        return h, coords


class DeltaEGNN(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int, message_dim: int, num_layers: int, coord_update_scale: float, delta_scale: float):
        super().__init__()
        self.input_proj = nn.Sequential(
            nn.LayerNorm(input_dim),
            nn.Linear(input_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
        )
        self.layers = nn.ModuleList([
            DeltaEGNNLayer(hidden_dim, message_dim, coord_update_scale) for _ in range(num_layers)
        ])
        self.out_mlp = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, 3),
        )
        self.delta_scale = delta_scale

    def forward(self, coords: torch.Tensor, esm: torch.Tensor, aa_one_hot: torch.Tensor, mask: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        h = self.input_proj(torch.cat([esm, aa_one_hot], dim=-1))
        coords_work = coords
        for layer in self.layers:
            h, coords_work = layer(h, coords_work, mask)
        pred_delta = torch.tanh(self.out_mlp(h)) * self.delta_scale * mask.unsqueeze(-1)
        pred_coords = coords_work + pred_delta
        return pred_delta, pred_coords


class DeltaTranslationCollator:
    def __call__(self, samples: list[dict[str, Any]]) -> dict[str, Any]:
        max_len = max(int(sample['residue_count']) for sample in samples)
        batch_size = len(samples)
        esm_dim = samples[0]['esm_embedding'].shape[-1]
        coords_source = torch.zeros(batch_size, max_len, 3)
        coords_target = torch.zeros(batch_size, max_len, 3)
        esm = torch.zeros(batch_size, max_len, esm_dim)
        aa = torch.zeros(batch_size, max_len, 21)
        mask = torch.zeros(batch_size, max_len)
        metadata: list[dict[str, Any]] = []
        for i, sample in enumerate(samples):
            n = int(sample['residue_count'])
            source_coords = structure_ca_coordinates(sample['source_structure']).astype(np.float32)
            target_coords = structure_ca_coordinates(sample['target_structure']).astype(np.float32)
            coords_source[i, :n] = torch.from_numpy(source_coords)
            coords_target[i, :n] = torch.from_numpy(target_coords)
            esm[i, :n] = sample['esm_embedding'].float()
            aa[i, :n] = one_hot_aatype(sample['source_structure']['aatype'])
            mask[i, :n] = 1.0
            metadata.append({
                'kinase': sample['kinase'],
                'source_pdb_id': sample['source_pdb_id'],
                'target_pdb_id': sample['target_pdb_id'],
                'residue_count': n,
                'sequence_identity': float(sample['sequence_identity']),
            })
        return {
            'coords_source': coords_source,
            'coords_target': coords_target,
            'esm': esm,
            'aa_one_hot': aa,
            'mask': mask,
            'metadata': metadata,
        }


def build_dataloaders(batch_size: int, num_workers: int) -> tuple[DataLoader, DataLoader, DataLoader]:
    pin_memory = device.type == 'cuda'
    collate_fn = DeltaTranslationCollator()
    common = dict(batch_size=batch_size, num_workers=num_workers, pin_memory=pin_memory, collate_fn=collate_fn, persistent_workers=num_workers > 0)
    return (
        DataLoader(train_ds, shuffle=True, **common),
        DataLoader(val_ds, shuffle=False, **common),
        DataLoader(test_ds, shuffle=False, **common),
    )


CONFIG = EGNNAnalysisConfig()
train_loader, val_loader, test_loader = build_dataloaders(CONFIG.batch_size, CONFIG.num_workers)

model_template = DeltaEGNN(
    input_dim=train_ds[0]['esm_embedding'].shape[-1] + 21,
    hidden_dim=CONFIG.hidden_dim,
    message_dim=CONFIG.message_dim,
    num_layers=CONFIG.num_layers,
    coord_update_scale=CONFIG.coord_update_scale,
    delta_scale=CONFIG.delta_scale,
).to(device)

print(CONFIG)
print('Trainable parameters:', sum(p.numel() for p in model_template.parameters() if p.requires_grad))
print('Total parameters:', sum(p.numel() for p in model_template.parameters()))
print('Memory after model init (MB):', round(process_memory_mb(), 2))

EGNNAnalysisConfig(fold_id=1, hidden_dim=96, message_dim=96, num_layers=3, batch_size=2, max_epochs=15, patience=4, learning_rate=5e-05, weight_decay=0.0001, grad_clip=1.0, num_workers=0, coord_update_scale=0.01, delta_scale=0.5, lambda_distance=0.05, use_amp=False)
Trainable parameters: 370768
Total parameters: 370768
Memory after model init (MB): 390.18


/Users/josefinadehan/tp_vision_avanzada/TPF_Vision_Avanzada/third_party/foldflow_official/openfold/utils/tensor_utils.py:74: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/autograd/python_variable_indexing.cpp:312.)
  return data[ranges]


In [5]:
def run_epoch(model: nn.Module, loader: DataLoader, optimizer: torch.optim.Optimizer | None, lambda_distance: float, use_amp: bool, grad_clip: float) -> dict[str, float]:
    training = optimizer is not None
    model.train(training)
    total_loss = 0.0
    total_delta_loss = 0.0
    total_distance_loss = 0.0
    total_rmsd_source_target = 0.0
    total_rmsd_prediction_target = 0.0
    total_rmsd_source_prediction = 0.0
    total_improvement = 0.0
    total_success = 0.0
    total_true_motion = 0.0
    total_pred_motion = 0.0
    total_samples = 0

    scaler = torch.amp.GradScaler('cuda', enabled=use_amp and device.type == 'cuda') if training and use_amp and device.type == 'cuda' else None

    for batch in loader:
        coords_source = batch['coords_source'].to(device)
        coords_target = batch['coords_target'].to(device)
        esm = batch['esm'].to(device)
        aa = batch['aa_one_hot'].to(device)
        mask = batch['mask'].to(device)
        true_delta = coords_target - coords_source

        with torch.amp.autocast('cuda', enabled=use_amp and device.type == 'cuda'):
            pred_delta, pred_coords = model(coords_source, esm, aa, mask)
            delta_loss = masked_mse(pred_delta, true_delta, mask)
            distance_loss = masked_pairwise_mse(pred_coords, coords_target, mask) if lambda_distance > 0 else torch.zeros((), device=device)
            loss = delta_loss + lambda_distance * distance_loss

        if not torch.isfinite(loss):
            print('Non-finite loss detected')
            print(tensor_health('coords_source', coords_source))
            print(tensor_health('coords_target', coords_target))
            print(tensor_health('pred_delta', pred_delta))
            print(tensor_health('pred_coords', pred_coords))
            print(tensor_health('true_delta', true_delta))
            raise FloatingPointError('Non-finite values during training')

        if training:
            optimizer.zero_grad(set_to_none=True)
            if scaler is not None:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                optimizer.step()
            if not math.isfinite(float(grad_norm)):
                raise FloatingPointError(f'Gradient norm became non-finite: {float(grad_norm)}')

        batch_size = coords_source.shape[0]
        for b in range(batch_size):
            n = int(mask[b].sum().item())
            source_np = coords_source[b, :n].detach().cpu().numpy()
            target_np = coords_target[b, :n].detach().cpu().numpy()
            pred_np = pred_coords[b, :n].detach().cpu().numpy()
            true_delta_np = true_delta[b, :n].detach().cpu().numpy()
            pred_delta_np = pred_delta[b, :n].detach().cpu().numpy()

            rmsd_source_target = rmsd_from_coordinates(target_np, source_np)
            rmsd_prediction_target = rmsd_from_coordinates(target_np, pred_np)
            rmsd_source_prediction = rmsd_from_coordinates(source_np, pred_np)
            improvement = rmsd_source_target - rmsd_prediction_target

            total_rmsd_source_target += rmsd_source_target
            total_rmsd_prediction_target += rmsd_prediction_target
            total_rmsd_source_prediction += rmsd_source_prediction
            total_improvement += improvement
            total_success += float(improvement > 0.0)
            total_true_motion += float(np.linalg.norm(true_delta_np, axis=-1).mean())
            total_pred_motion += float(np.linalg.norm(pred_delta_np, axis=-1).mean())
            total_samples += 1

        total_loss += float(loss.detach().cpu()) * batch_size
        total_delta_loss += float(delta_loss.detach().cpu()) * batch_size
        total_distance_loss += float(distance_loss.detach().cpu()) * batch_size

    denom = max(1, total_samples)
    return {
        'total_loss': total_loss / denom,
        'delta_loss': total_delta_loss / denom,
        'distance_loss': total_distance_loss / denom,
        'rmsd_source_target': total_rmsd_source_target / denom,
        'rmsd_prediction_target': total_rmsd_prediction_target / denom,
        'rmsd_source_prediction': total_rmsd_source_prediction / denom,
        'rmsd_improvement': total_improvement / denom,
        'success_rate': total_success / denom,
        'mean_true_motion': total_true_motion / denom,
        'mean_pred_motion': total_pred_motion / denom,
        'mean_delta_norm': total_pred_motion / denom,
        'mean_true_delta_norm': total_true_motion / denom,
        'num_samples': float(total_samples),
    }


def predict_loader(model: nn.Module, loader: DataLoader) -> pd.DataFrame:
    model.eval()
    rows: list[dict[str, Any]] = []
    with torch.no_grad():
        for sample_index, batch in enumerate(loader):
            coords_source = batch['coords_source'].to(device)
            coords_target = batch['coords_target'].to(device)
            esm = batch['esm'].to(device)
            aa = batch['aa_one_hot'].to(device)
            mask = batch['mask'].to(device)
            true_delta = coords_target - coords_source
            pred_delta, pred_coords = model(coords_source, esm, aa, mask)

            for b in range(coords_source.shape[0]):
                n = int(mask[b].sum().item())
                source_np = coords_source[b, :n].cpu().numpy()
                target_np = coords_target[b, :n].cpu().numpy()
                pred_np = pred_coords[b, :n].cpu().numpy()
                true_delta_np = true_delta[b, :n].cpu().numpy()
                pred_delta_np = pred_delta[b, :n].cpu().numpy()
                row_meta = batch['metadata'][b]
                true_motion = np.linalg.norm(true_delta_np, axis=-1)
                pred_motion = np.linalg.norm(pred_delta_np, axis=-1)
                rows.append({
                    'sample_index': sample_index * loader.batch_size + b,
                    'kinase': row_meta['kinase'],
                    'source_pdb_id': row_meta['source_pdb_id'],
                    'target_pdb_id': row_meta['target_pdb_id'],
                    'residue_count': n,
                    'sequence_identity': row_meta['sequence_identity'],
                    'rmsd_source_target': rmsd_from_coordinates(target_np, source_np),
                    'rmsd_prediction_target': rmsd_from_coordinates(target_np, pred_np),
                    'rmsd_source_prediction': rmsd_from_coordinates(source_np, pred_np),
                    'rmsd_improvement': rmsd_from_coordinates(target_np, source_np) - rmsd_from_coordinates(target_np, pred_np),
                    'success': rmsd_from_coordinates(target_np, pred_np) < rmsd_from_coordinates(target_np, source_np),
                    'mean_true_motion': float(true_motion.mean()),
                    'mean_pred_motion': float(pred_motion.mean()),
                    'mean_delta_norm': float(np.linalg.norm(pred_delta_np, axis=-1).mean()),
                    'mean_true_delta_norm': float(np.linalg.norm(true_delta_np, axis=-1).mean()),
                    'motion_correlation': motion_correlation(true_motion, pred_motion),
                    'source_coords': source_np.tolist(),
                    'target_coords': target_np.tolist(),
                    'prediction_coords': pred_np.tolist(),
                    'true_delta': true_delta_np.tolist(),
                    'pred_delta': pred_delta_np.tolist(),
                    'true_motion': true_motion.tolist(),
                    'pred_motion': pred_motion.tolist(),
                    'residue_error': np.linalg.norm(pred_np - target_np, axis=-1).tolist(),
                })
    return pd.DataFrame(rows)


def train_experiment(experiment_name: str, lambda_distance: float) -> dict[str, Any]:
    config = EGNNAnalysisConfig(lambda_distance=lambda_distance)
    train_loader, val_loader, test_loader = build_dataloaders(config.batch_size, config.num_workers)
    model = DeltaEGNN(
        input_dim=train_ds[0]['esm_embedding'].shape[-1] + 21,
        hidden_dim=config.hidden_dim,
        message_dim=config.message_dim,
        num_layers=config.num_layers,
        coord_update_scale=config.coord_update_scale,
        delta_scale=config.delta_scale,
    ).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)
    checkpoint_dir = CHECKPOINT_ROOT / experiment_name
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    best_checkpoint = checkpoint_dir / 'best.pt'
    last_checkpoint = checkpoint_dir / 'last.pt'

    history: list[dict[str, Any]] = []
    best_val = float('inf')
    best_epoch = 0
    patience = 0
    started = time.perf_counter()
    for epoch in tqdm(range(1, config.max_epochs + 1), desc=f'Training {experiment_name}'):
        train_metrics = run_epoch(model, train_loader, optimizer, lambda_distance, config.use_amp, config.grad_clip)
        val_metrics = run_epoch(model, val_loader, None, lambda_distance, False, config.grad_clip)
        row = {'epoch': epoch, **{f'train_{k}': v for k, v in train_metrics.items()}, **{f'val_{k}': v for k, v in val_metrics.items()}}
        history.append(row)
        torch.save({'epoch': epoch, 'model_state': model.state_dict(), 'optimizer_state': optimizer.state_dict(), 'config': config.__dict__}, last_checkpoint)
        if val_metrics['total_loss'] < best_val:
            best_val = val_metrics['total_loss']
            best_epoch = epoch
            patience = 0
            torch.save({'epoch': epoch, 'model_state': model.state_dict(), 'optimizer_state': optimizer.state_dict(), 'config': config.__dict__}, best_checkpoint)
        else:
            patience += 1
        print(f"[{experiment_name}] epoch {epoch:03d} train_loss={train_metrics['total_loss']:.4f} val_loss={val_metrics['total_loss']:.4f} val_rmsd={val_metrics['rmsd_prediction_target']:.3f} improv={val_metrics['rmsd_improvement']:.3f}")
        if patience >= config.patience:
            print(f'[{experiment_name}] early stopping at epoch {epoch} (best epoch {best_epoch})')
            break

    training_seconds = time.perf_counter() - started
    history_frame = pd.DataFrame(history)
    history_path = REPORT_ROOT / f'{experiment_name}_training_history.csv'
    save_csv(history_path, history_frame)
    test_frame = predict_loader(model, test_loader)
    test_path = REPORT_ROOT / f'{experiment_name}_test_predictions.csv'
    test_slim = test_frame[[
        'sample_index', 'kinase', 'source_pdb_id', 'target_pdb_id', 'residue_count', 'sequence_identity',
        'rmsd_source_target', 'rmsd_prediction_target', 'rmsd_source_prediction', 'rmsd_improvement',
        'success', 'mean_true_motion', 'mean_pred_motion', 'mean_delta_norm', 'mean_true_delta_norm',
        'motion_correlation',
    ]].copy()
    save_csv(test_path, test_slim)
    val_frame = predict_loader(model, val_loader)
    val_path = REPORT_ROOT / f'{experiment_name}_validation_predictions.csv'
    val_slim = val_frame[[
        'sample_index', 'kinase', 'source_pdb_id', 'target_pdb_id', 'residue_count', 'sequence_identity',
        'rmsd_source_target', 'rmsd_prediction_target', 'rmsd_source_prediction', 'rmsd_improvement',
        'success', 'mean_true_motion', 'mean_pred_motion', 'mean_delta_norm', 'mean_true_delta_norm',
        'motion_correlation',
    ]].copy()
    save_csv(val_path, val_slim)

    summary = {
        'experiment': experiment_name,
        'lambda_distance': lambda_distance,
        'best_epoch': best_epoch,
        'best_validation_loss': best_val,
        'final_train_loss': float(history_frame['train_total_loss'].iloc[-1]),
        'final_validation_loss': float(history_frame['val_total_loss'].iloc[-1]),
        'train_size': len(train_ds),
        'validation_size': len(val_ds),
        'test_size': len(test_ds),
        'trainable_parameters': int(sum(p.numel() for p in model.parameters() if p.requires_grad)),
        'total_parameters': int(sum(p.numel() for p in model.parameters())),
        'training_seconds': float(training_seconds),
        'checkpoint_best': str(best_checkpoint),
        'checkpoint_last': str(last_checkpoint),
        'history_csv': str(history_path),
        'validation_predictions_csv': str(val_path),
        'test_predictions_csv': str(test_path),
        'model_state': model.state_dict(),
        'history_frame': history_frame,
        'test_frame': test_frame,
        'val_frame': val_frame,
    }
    return summary

In [6]:
experiments = [
    ('EGNN_delta_only', 0.0),
    ('EGNN_delta_plus_distance', 0.05),
]

experiment_runs: dict[str, dict[str, Any]] = {}
for name, lambda_distance in experiments:
    experiment_runs[name] = train_experiment(name, lambda_distance)

print('Completed experiments:', list(experiment_runs))

Training EGNN_delta_only:   0%|          | 0/15 [00:00<?, ?it/s]/Users/josefinadehan/tp_vision_avanzada/TPF_Vision_Avanzada/third_party/foldflow_official/openfold/utils/tensor_utils.py:74: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/autograd/python_variable_indexing.cpp:312.)
  return data[ranges]
Training EGNN_delta_only:   7%|▋         | 1/15 [03:05<43:14, 185.31s/it]

[EGNN_delta_only] epoch 001 train_loss=8.1807 val_loss=7.6965 val_rmsd=3.074 improv=0.004


Training EGNN_delta_only:  13%|█▎        | 2/15 [05:48<37:19, 172.23s/it]

[EGNN_delta_only] epoch 002 train_loss=7.9645 val_loss=7.6978 val_rmsd=3.087 improv=-0.009


Training EGNN_delta_only:  20%|██        | 3/15 [08:31<33:38, 168.19s/it]

...

[EGNN_delta_plus_distance] epoch 006 train_loss=7.9865 val_loss=8.0039 val_rmsd=3.075 improv=0.003


Training EGNN_delta_plus_distance:  40%|████      | 6/15 [19:54<29:51, 199.10s/it]

[EGNN_delta_plus_distance] epoch 007 train_loss=7.8651 val_loss=7.9682 val_rmsd=3.080 improv=-0.003
[EGNN_delta_plus_distance] early stopping at epoch 7 (best epoch 3)



/Users/josefinadehan/tp_vision_avanzada/TPF_Vision_Avanzada/third_party/foldflow_official/openfold/utils/tensor_utils.py:74: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/autograd/python_variable_indexing.cpp:312.)
  return data[ranges]
/Users/josefinadehan/tp_vision_avanzada/TPF_Vision_Avanzada/third_party/foldflow_official/openfold/utils/tensor_utils.py:74: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered int

Completed experiments: ['EGNN_delta_only', 'EGNN_delta_plus_distance']


In [7]:
def _mean_scalar(frame: pd.DataFrame, column: str) -> float:
    values = frame[column]
    if isinstance(values, pd.DataFrame):
        values = values.iloc[:, 0]
    return float(values.mean())


def split_metric_table(frame: pd.DataFrame, method_name: str) -> dict[str, Any]:
    return {
        'Method': method_name,
        'RMSD Pred→Target': _mean_scalar(frame, 'rmsd_prediction_target'),
        'RMSD Source→Target': _mean_scalar(frame, 'rmsd_source_target'),
        'RMSD Source→Pred': _mean_scalar(frame, 'rmsd_source_prediction'),
        'Improvement': _mean_scalar(frame, 'rmsd_improvement'),
        'Success Rate': _mean_scalar(frame, 'success'),
        'Mean True Motion': _mean_scalar(frame, 'mean_true_motion'),
        'Mean Pred Motion': _mean_scalar(frame, 'mean_pred_motion'),
        'Mean Delta Norm': _mean_scalar(frame, 'mean_delta_norm'),
        'Mean True Delta Norm': _mean_scalar(frame, 'mean_true_delta_norm'),
    }


identity_metrics = split_metric_table(identity_test, 'Identity')
identity_metrics['Mean Pred Motion'] = 0.0
identity_metrics['Mean Delta Norm'] = 0.0

comparison_rows = [identity_metrics]
for name in experiments:
    exp = experiment_runs[name[0]]
    test_frame = exp['test_frame']
    comparison_rows.append(split_metric_table(test_frame, name[0]))

comparison_table = pd.DataFrame(comparison_rows)
display(comparison_table)
save_csv(REPORT_ROOT / 'fold1_egnn_method_comparison.csv', comparison_table)

# Choose the stronger model for residue-level analysis.
best_experiment_name = max(
    [name for name, _ in experiments],
    key=lambda n: experiment_runs[n]['val_frame']['rmsd_prediction_target'].mean(),
)
if len(experiments) > 1:
    best_experiment_name = min(
        [name for name, _ in experiments],
        key=lambda n: experiment_runs[n]['val_frame']['rmsd_prediction_target'].mean(),
    )

best_run = experiment_runs[best_experiment_name]
print('Selected analysis model:', best_experiment_name)
print('Validation RMSD:', best_run['val_frame']['rmsd_prediction_target'].mean())
print('Test RMSD:', best_run['test_frame']['rmsd_prediction_target'].mean())

if foldflow_reference is not None:
    print(f'Reference FoldFlow conditional RMSD: {foldflow_reference:.3f} Å')

TypeError: float() argument must be a string or a real number, not 'Series'

In [8]:
def plot_history(history: pd.DataFrame, title: str, output_path: Path) -> Path:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history['epoch'], history['train_total_loss'], label='train')
    axes[0].plot(history['epoch'], history['val_total_loss'], label='validation')
    axes[0].set_title(f'{title} total loss')
    axes[0].set_xlabel('epoch')
    axes[0].set_ylabel('loss')
    axes[0].legend()

    axes[1].plot(history['epoch'], history['train_delta_loss'], label='train delta')
    axes[1].plot(history['epoch'], history['val_delta_loss'], label='val delta')
    axes[1].plot(history['epoch'], history['train_distance_loss'], label='train distance')
    axes[1].plot(history['epoch'], history['val_distance_loss'], label='val distance')
    axes[1].set_title(f'{title} component losses')
    axes[1].set_xlabel('epoch')
    axes[1].legend()
    fig.tight_layout()
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_path, dpi=180, bbox_inches='tight')
    plt.close(fig)
    return output_path


def plot_rmsd_curves(history: pd.DataFrame, title: str, output_path: Path) -> Path:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history['epoch'], history['train_rmsd_prediction_target'], label='train pred→target')
    axes[0].plot(history['epoch'], history['val_rmsd_prediction_target'], label='validation pred→target')
    axes[0].plot(history['epoch'], history['train_rmsd_source_target'], label='train source→target', linestyle='--')
    axes[0].plot(history['epoch'], history['val_rmsd_source_target'], label='validation source→target', linestyle='--')
    axes[0].set_title(f'{title} RMSD curves')
    axes[0].set_xlabel('epoch')
    axes[0].set_ylabel('RMSD (Å)')
    axes[0].legend(fontsize=8)

    axes[1].plot(history['epoch'], history['train_rmsd_improvement'], label='train improvement')
    axes[1].plot(history['epoch'], history['val_rmsd_improvement'], label='validation improvement')
    axes[1].axhline(0.0, color='black', linestyle='--', linewidth=1)
    axes[1].set_title(f'{title} improvement')
    axes[1].set_xlabel('epoch')
    axes[1].set_ylabel('RMSD gain (Å)')
    axes[1].legend(fontsize=8)
    fig.tight_layout()
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_path, dpi=180, bbox_inches='tight')
    plt.close(fig)
    return output_path


def plot_improvement_hist(frame: pd.DataFrame, output_path: Path, title: str) -> Path:
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(frame['rmsd_improvement'], bins=25, color='#1f77b4', alpha=0.85, edgecolor='white')
    ax.axvline(0.0, color='black', linestyle='--', linewidth=1)
    ax.set_title(title)
    ax.set_xlabel('RMSD improvement (Å)')
    ax.set_ylabel('count')
    fig.tight_layout()
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_path, dpi=180, bbox_inches='tight')
    plt.close(fig)
    return output_path


def plot_motion_correlation(frame: pd.DataFrame, output_path: Path, title: str) -> Path:
    correlations = frame['motion_correlation'].replace([np.inf, -np.inf], np.nan).dropna()
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(correlations, bins=20, color='#ff7f0e', alpha=0.85, edgecolor='white')
    ax.set_title(title)
    ax.set_xlabel('Pearson correlation')
    ax.set_ylabel('count')
    fig.tight_layout()
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_path, dpi=180, bbox_inches='tight')
    plt.close(fig)
    return output_path


def plot_motion_overlay(true_motion: np.ndarray, pred_motion: np.ndarray, title: str, output_path: Path) -> Path:
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(true_motion, label='true motion', color='#2ca02c', linewidth=2)
    ax.plot(pred_motion, label='predicted motion', color='#d62728', linewidth=2)
    ax.set_title(title)
    ax.set_xlabel('residue index')
    ax.set_ylabel('motion magnitude (Å)')
    ax.legend()
    fig.tight_layout()
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_path, dpi=180, bbox_inches='tight')
    plt.close(fig)
    return output_path


def plot_residue_error(error: np.ndarray, title: str, output_path: Path) -> Path:
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(error, color='#9467bd', linewidth=2)
    ax.set_title(title)
    ax.set_xlabel('residue index')
    ax.set_ylabel('prediction error (Å)')
    fig.tight_layout()
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_path, dpi=180, bbox_inches='tight')
    plt.close(fig)
    return output_path


def plot_structure_overlay(source_coords: np.ndarray, pred_coords: np.ndarray, target_coords: np.ndarray, title: str, output_path: Path) -> Path:
    fig = plt.figure(figsize=(15, 5))
    axes = [fig.add_subplot(1, 3, i + 1, projection='3d') for i in range(3)]
    for ax, (label, coords, color) in zip(
        axes,
        [
            ('Inactive source', source_coords, '#8c8c8c'),
            ('Predicted active', pred_coords, '#d62728'),
            ('Real active', target_coords, '#2ca02c'),
        ],
    ):
        ax.plot(coords[:, 0], coords[:, 1], coords[:, 2], color=color, linewidth=1.5)
        ax.scatter(coords[:, 0], coords[:, 1], coords[:, 2], c=color, s=8)
        ax.set_title(label)
        ax.set_axis_off()
    fig.suptitle(title)
    fig.tight_layout()
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_path, dpi=180, bbox_inches='tight')
    plt.close(fig)
    return output_path


def plot_colored_structure(coords: np.ndarray, values: np.ndarray, title: str, output_path: Path, cmap: str = 'coolwarm') -> Path:
    fig = plt.figure(figsize=(7, 6))
    ax = fig.add_subplot(111, projection='3d')
    sc = ax.scatter(coords[:, 0], coords[:, 1], coords[:, 2], c=values, cmap=cmap, s=18)
    ax.plot(coords[:, 0], coords[:, 1], coords[:, 2], color='black', alpha=0.25, linewidth=1)
    ax.set_title(title)
    ax.set_axis_off()
    fig.colorbar(sc, ax=ax, shrink=0.7, pad=0.05)
    fig.tight_layout()
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_path, dpi=180, bbox_inches='tight')
    plt.close(fig)
    return output_path


def plot_high_motion_residues(coords: np.ndarray, true_motion: np.ndarray, pred_motion: np.ndarray, threshold: float, title: str, output_path: Path) -> Path:
    mask = true_motion > threshold
    fig = plt.figure(figsize=(12, 5))
    axes = [fig.add_subplot(1, 2, i + 1, projection='3d') for i in range(2)]
    for ax, values, label in zip(axes, [true_motion, pred_motion], ['true motion', 'predicted motion']):
        ax.plot(coords[:, 0], coords[:, 1], coords[:, 2], color='lightgrey', alpha=0.4, linewidth=1)
        if mask.any():
            sc = ax.scatter(coords[mask, 0], coords[mask, 1], coords[mask, 2], c=values[mask], cmap='coolwarm', s=24)
            fig.colorbar(sc, ax=ax, shrink=0.6, pad=0.02)
        ax.set_title(label)
        ax.set_axis_off()
    fig.suptitle(title)
    fig.tight_layout()
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_path, dpi=180, bbox_inches='tight')
    plt.close(fig)
    return output_path

In [13]:
# Training histories and prediction tables are already stored in experiment_runs.
for name, payload in experiment_runs.items():
    history = payload['history_frame']
    plot_history(history, name, FIGURE_ROOT / f'{name}_training_curves.png')
    plot_rmsd_curves(history, name, FIGURE_ROOT / f'{name}_rmsd_curves.png')
    plot_improvement_hist(payload['test_frame'], FIGURE_ROOT / f'{name}_test_improvement_hist.png', f'{name} test RMSD improvement')
    plot_motion_correlation(payload['test_frame'], FIGURE_ROOT / f'{name}_motion_correlation_hist.png', f'{name} motion correlation')

# Per-residue analysis uses the selected model.
best_experiment_name = min(
    experiment_runs,
    key=lambda n: experiment_runs[n]['val_frame']['rmsd_prediction_target'].mean(),
)
best_run = experiment_runs[best_experiment_name]
print('Selected analysis model (cell 9):', best_experiment_name)

def _identity_summary_from_frame(frame: pd.DataFrame) -> dict[str, Any]:
    return {
        'Method': 'Identity',
        'RMSD Pred→Target': float(frame['rmsd_prediction_target'].mean()),
        'RMSD Source→Target': float(frame['rmsd_source_target'].mean()),
        'RMSD Source→Pred': float(frame['rmsd_source_prediction'].mean()),
        'Improvement': float(frame['rmsd_improvement'].mean()),
        'Success Rate': float(frame['success'].mean()),
        'Mean True Motion': float(frame['mean_true_motion'].mean()),
        'Mean Pred Motion': 0.0,
        'Mean Delta Norm': 0.0,
        'Mean True Delta Norm': float(frame['mean_true_delta_norm'].mean()),
    }

identity_metrics = _identity_summary_from_frame(identity_test)

test_frame = best_run['test_frame'].sort_values('rmsd_improvement', ascending=False).reset_index(drop=True)
selected_examples = {
    'best': test_frame.iloc[0],
    'median': test_frame.iloc[len(test_frame) // 2],
    'worst': test_frame.iloc[-1],
}

example_root = FIGURE_ROOT / 'examples'
example_root.mkdir(parents=True, exist_ok=True)
for label, row in selected_examples.items():
    source_coords = np.asarray(row['source_coords'], dtype=np.float64)
    pred_coords = np.asarray(row['prediction_coords'], dtype=np.float64)
    target_coords = np.asarray(row['target_coords'], dtype=np.float64)
    true_motion = np.asarray(row['true_motion'], dtype=np.float64)
    pred_motion = np.asarray(row['pred_motion'], dtype=np.float64)
    residue_error = np.asarray(row['residue_error'], dtype=np.float64)

    plot_structure_overlay(source_coords, pred_coords, target_coords, f'{label.title()} sample | {row["kinase"]}', example_root / f'{label}_overlay.png')
    plot_motion_overlay(true_motion, pred_motion, f'{label.title()} sample motion profile | {row["kinase"]}', example_root / f'{label}_motion_overlay.png')
    plot_residue_error(residue_error, f'{label.title()} sample residue error | {row["kinase"]}', example_root / f'{label}_residue_error.png')
    plot_colored_structure(source_coords, pred_motion, f'{label.title()} sample predicted motion | {row["kinase"]}', example_root / f'{label}_predicted_motion_structure.png')
    plot_colored_structure(source_coords, true_motion, f'{label.title()} sample true motion | {row["kinase"]}', example_root / f'{label}_true_motion_structure.png')
    plot_high_motion_residues(source_coords, true_motion, pred_motion, threshold=2.0, title=f'{label.title()} sample high-motion residues | {row["kinase"]}', output_path=example_root / f'{label}_high_motion_residues.png')

# Distribution of per-sample correlations and a compact summary table.
correlation_frame = best_run['test_frame'][['kinase', 'residue_count', 'motion_correlation', 'rmsd_improvement', 'rmsd_prediction_target']].copy()
save_csv(REPORT_ROOT / 'fold1_egnn_motion_correlations.csv', correlation_frame)

summary = {
    'dataset': {
        'train_size': len(train_ds),
        'validation_size': len(val_ds),
        'test_size': len(test_ds),
        'train_average_rmsd_source_target': float(train_summary['rmsd_source_target'].mean()),
        'validation_average_rmsd_source_target': float(val_summary['rmsd_source_target'].mean()),
        'test_average_rmsd_source_target': float(test_summary['rmsd_source_target'].mean()),
    },
    'identity': identity_metrics,
    'experiments': {},
    'foldflow_reference_conditional_rmsd': foldflow_reference,
}

for name, payload in experiment_runs.items():
    test_frame = payload['test_frame']
    val_frame = payload['val_frame']
    summary['experiments'][name] = {
        'best_epoch': int(pd.DataFrame(payload['history_frame'])['val_total_loss'].idxmin() + 1),
        'best_validation_loss': float(payload['history_frame']['val_total_loss'].min()),
        'final_training_loss': float(payload['history_frame']['train_total_loss'].iloc[-1]),
        'final_validation_loss': float(payload['history_frame']['val_total_loss'].iloc[-1]),
        'validation_rmsd_prediction_target': float(val_frame['rmsd_prediction_target'].mean()),
        'validation_improvement': float(val_frame['rmsd_improvement'].mean()),
        'validation_success_rate': float(val_frame['success'].mean()),
        'test_rmsd_prediction_target': float(test_frame['rmsd_prediction_target'].mean()),
        'test_rmsd_source_target': float(test_frame['rmsd_source_target'].mean()),
        'test_improvement': float(test_frame['rmsd_improvement'].mean()),
        'test_success_rate': float(test_frame['success'].mean()),
        'mean_true_motion': float(test_frame['mean_true_motion'].mean()),
        'mean_pred_motion': float(test_frame['mean_pred_motion'].mean()),
        'mean_delta_norm': float(test_frame['mean_delta_norm'].mean()),
        'mean_true_delta_norm': float(test_frame['mean_true_delta_norm'].mean()),
        'training_seconds': float(payload['training_seconds']),
        'checkpoint_best': payload['checkpoint_best'],
        'checkpoint_last': payload['checkpoint_last'],
    }

comparison_rows = [identity_metrics]
for name, payload in experiment_runs.items():
    test_frame = payload['test_frame']
    comparison_rows.append({
        'Method': name,
        'RMSD Pred→Target': float(test_frame['rmsd_prediction_target'].mean()),
        'RMSD Source→Target': float(test_frame['rmsd_source_target'].mean()),
        'RMSD Source→Pred': float(test_frame['rmsd_source_prediction'].mean()),
        'Improvement': float(test_frame['rmsd_improvement'].mean()),
        'Success Rate': float(test_frame['success'].mean()),
        'Mean True Motion': float(test_frame['mean_true_motion'].mean()),
        'Mean Pred Motion': float(test_frame['mean_pred_motion'].mean()),
        'Mean Delta Norm': float(test_frame['mean_delta_norm'].mean()),
        'Mean True Delta Norm': float(test_frame['mean_true_delta_norm'].mean()),
    })
comparison_table = pd.DataFrame(comparison_rows)

comparison_plot = comparison_table.copy()
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
bar_colors = ['#7f7f7f', '#1f77b4', '#ff7f0e']
axes[0].bar(comparison_plot['Method'], comparison_plot['RMSD Pred→Target'], color=bar_colors[:len(comparison_plot)])
axes[0].set_title('Test RMSD pred→target')
axes[0].tick_params(axis='x', rotation=20)
axes[1].bar(comparison_plot['Method'], comparison_plot['Improvement'], color=bar_colors[:len(comparison_plot)])
axes[1].set_title('Test RMSD improvement')
axes[1].tick_params(axis='x', rotation=20)
axes[2].bar(comparison_plot['Method'], comparison_plot['Success Rate'], color=bar_colors[:len(comparison_plot)])
axes[2].set_title('Success rate')
axes[2].tick_params(axis='x', rotation=20)
fig.tight_layout()
fig_path = FIGURE_ROOT / 'method_comparison.png'
fig_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(fig_path, dpi=180, bbox_inches='tight')
plt.close(fig)
print('Saved method comparison figure to', fig_path)

if len(experiment_runs) >= 2:
    names = list(experiment_runs)
    summary['comparisons'] = {
        'EGNN_delta_only_vs_identity': float(comparison_table.loc[comparison_table['Method'] == 'EGNN_delta_only', 'RMSD Pred→Target'].iloc[0] - comparison_table.loc[comparison_table['Method'] == 'Identity', 'RMSD Pred→Target'].iloc[0]),
        'EGNN_delta_plus_distance_vs_identity': float(comparison_table.loc[comparison_table['Method'] == 'EGNN_delta_plus_distance', 'RMSD Pred→Target'].iloc[0] - comparison_table.loc[comparison_table['Method'] == 'Identity', 'RMSD Pred→Target'].iloc[0]),
    }

summary_path = REPORT_ROOT / 'fold1_egnn_analysis_summary.json'
safe_json_write(summary_path, summary)
print('Wrote summary to', summary_path)
print('\nFinal comparison table:')
display(comparison_table)

Selected analysis model (cell 9): EGNN_delta_plus_distance
Saved method comparison figure to /Users/josefinadehan/tp_vision_avanzada/TPF_Vision_Avanzada/figures/fold1_egnn/method_comparison.png
Wrote summary to /Users/josefinadehan/tp_vision_avanzada/TPF_Vision_Avanzada/reports/fold1_egnn/fold1_egnn_analysis_summary.json

Final comparison table:


,Method,RMSD Pred→Target,RMSD Source→Target,RMSD Source→Pred,Improvement,Success Rate,Mean True Motion,Mean Pred Motion,Mean Delta Norm,Mean True Delta Norm
0,Identity,3.316142,3.316142,0.00000,0.000000,0.000000,2.253116,0.000000,0.000000,2.253116
1,EGNN_delta_only,3.355289,3.316142,0.75282,-0.039148,0.495050,2.253116,0.652758,0.652758,2.253116
2,EGNN_delta_plus_distance,3.304522,3.316142,0.50493,0.011619,0.762376,2.253116,0.627350,0.627350,2.253116
